<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/14C_NeuroFHIR_Review_WISH_End_to_End_Dry_Run_and_Instrumentation_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 — Load Notebook 14B final package
from __future__ import annotations
import csv, json, re, subprocess, time
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT=Path('/content/drive/MyDrive/neurofhir-qc')
FINAL_ROOT=PROJECT_ROOT/'wish_extension/final_wish_pilot'
PARTICIPANT_ROOT=FINAL_ROOT/'participant_app'
FINAL_HTML=PARTICIPANT_ROOT/'index.html'
PARTICIPANT_JSON=PARTICIPANT_ROOT/'participant_cases.json'
RESEARCH_ROOT=FINAL_ROOT/'researcher_only'
RESEARCH_KEY=RESEARCH_ROOT/'final_researcher_scenario_key.json'
ALLOCATION_CSV=RESEARCH_ROOT/'blocked_sequence_allocation.csv'
EVAL_ROOT=FINAL_ROOT/'evaluation'
NB14B_AUDIT=EVAL_ROOT/'notebook_14b_final_pilot_readiness_audit.json'
DRY_ROOT=EVAL_ROOT/'dry_run'
DRY_ROOT.mkdir(parents=True,exist_ok=True)

def now(): return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace('+00:00','Z')
def loadj(p:Path)->Any:
    with p.open('r',encoding='utf-8') as f: return json.load(f)
def writej(p:Path,obj:Any):
    p.parent.mkdir(parents=True,exist_ok=True)
    with p.open('w',encoding='utf-8') as f:
        json.dump(obj,f,indent=2,ensure_ascii=False,allow_nan=False); f.write('\n')

required=[FINAL_HTML,PARTICIPANT_JSON,RESEARCH_KEY,ALLOCATION_CSV,NB14B_AUDIT]
missing=[str(p) for p in required if not p.exists() or p.stat().st_size==0]
if missing: raise FileNotFoundError('Notebook 14B package incomplete:\n'+'\n'.join(missing))
audit14b=loadj(NB14B_AUDIT)
if audit14b.get('status')!='completed' or not audit14b.get('final_gate'): raise RuntimeError('Notebook 14B did not pass final gate.')
participant=loadj(PARTICIPANT_JSON); research=loadj(RESEARCH_KEY)
assert len(participant['cases'])==12 and len(research['cases'])==12
print('✅ Notebook 14B final gate confirmed')
print('✅ 12 participant cases loaded')


Mounted at /content/drive
✅ Notebook 14B final gate confirmed
✅ 12 participant cases loaded


In [2]:
# Cell 2 — Correct AI-exposure event timing in the final app
html=FINAL_HTML.read_text(encoding='utf-8')
pattern=(
 r'function nextScreen\(n\)\{\s*'
 r'S\.screen=n;render\(\);log\("screen_opened"\);\s*'
 r'if\(n===1\)log\("evidence_opened"\);\s*'
 r'if\(n===3\)log\("ai_exposed",\{ai_visible_before_initial_judgment:condition\(\)==="ai-first"\}\);\s*'
 r'if\(n===4\)log\("passport_opened"\);\s*\}'
)
replacement=(
 'function nextScreen(n){\n'
 ' S.screen=n;render();log("screen_opened");\n'
 ' if(n===1){\n'
 '  log("evidence_opened");\n'
 '  if(condition()==="ai-first"){\n'
 '   log("ai_exposed",{ai_visible_before_initial_judgment:true});\n'
 '  }\n'
 ' }\n'
 ' if(n===3 && condition()==="evidence-first"){\n'
 '  log("ai_exposed",{ai_visible_before_initial_judgment:false});\n'
 ' }\n'
 ' if(n===4)log("passport_opened");\n'
 '}'
)
patched,count=re.subn(pattern,lambda m:replacement,html,count=1,flags=re.MULTILINE|re.DOTALL)
if count==0:
    checks=['if(n===1){','ai_visible_before_initial_judgment:true','if(n===3 && condition()==="evidence-first")','ai_visible_before_initial_judgment:false']
    if not all(x in html for x in checks): raise RuntimeError('Could not locate or verify nextScreen() instrumentation block.')
    patched=html; print('ℹ️ Corrected instrumentation already present')
else: print('✅ AI-exposure timing patched')
old='if(n===3)log("ai_exposed",{ai_visible_before_initial_judgment:condition()==="ai-first"});'
assert old not in patched
FINAL_HTML.write_text(patched,encoding='utf-8')
writej(DRY_ROOT/'interaction_instrumentation_patch.json',{'generated_utc':now(),'status':'passed','ai_first':'exposure before initial judgment','evidence_first':'exposure after initial judgment','one_ai_exposure_per_case':True})
print('✅ Interaction instrumentation gate passed')


✅ AI-exposure timing patched
✅ Interaction instrumentation gate passed


In [3]:
# Cell 3 — Validate package, blocked allocation, and condition balance
cases=participant['cases']; research_cases=research['cases']
assert len({c['case_id'] for c in cases})==12 and len({c['scenario_id'] for c in cases})==12
missing_assets=[]
for c in cases:
    p=PARTICIPANT_ROOT/c['preview_file']
    if not p.exists() or p.stat().st_size==0: missing_assets.append(str(p))
assert not missing_assets,missing_assets
bad=[str(p) for p in PARTICIPANT_ROOT.rglob('*') if p.is_file() and ('reference' in p.name.lower() or p.name.lower().endswith('.nii') or p.name.lower().endswith('.nii.gz') or 'researcher' in p.name.lower())]
assert not bad,bad
allocation={}
with ALLOCATION_CSV.open('r',encoding='utf-8',newline='') as f:
    for row in csv.DictReader(f): allocation[row['participant_id']]=row['hidden_sequence']
assert allocation['P001']=='A' and allocation['P002']=='B'
rmap={c['scenario_id']:c for c in research_cases}
def condition_for(sid,seq): return rmap[sid]['condition_by_sequence'][seq]
for seq in ('A','B'):
    counts={'evidence-first':0,'ai-first':0}; matrix={'evidence-first':{'stable':0,'progression':0},'ai-first':{'stable':0,'progression':0}}
    for c in research_cases:
        cond=condition_for(c['scenario_id'],seq); counts[cond]+=1; matrix[cond][c['trajectory']]+=1
    assert counts=={'evidence-first':6,'ai-first':6}
    assert matrix=={'evidence-first':{'stable':3,'progression':3},'ai-first':{'stable':3,'progression':3}}
print('✅ Package + allocation + balance validation passed')
print('✅ P001=A, P002=B; each sequence has 6/6 and 3 stable + 3 progression per condition')


✅ Package + allocation + balance validation passed
✅ P001=A, P002=B; each sequence has 6/6 and 3 stable + 3 progression per condition


In [4]:
# Cell 4 — Simulate complete P001 and P002 sessions
def fnv1a(text):
    h=2166136261
    for ch in text: h=((h^ord(ch))*16777619)&0xffffffff
    return h
def app_shuffle(items,seed):
    out=list(items); state=fnv1a(seed)
    def rand():
        nonlocal state; state=(state*1664525+1013904223)&0xffffffff; return state/4294967296
    for i in range(len(out)-1,0,-1):
        j=int(rand()*(i+1)); out[i],out[j]=out[j],out[i]
    return out
def iso(base,t): return (base+timedelta(seconds=t)).replace(microsecond=0).isoformat().replace('+00:00','Z')

def simulate(pid):
    seq=allocation[pid]; order=app_shuffle(cases,'order-'+pid); base=datetime(2026,9,10,12,0,0,tzinfo=timezone.utc); tick=0; events=[]; responses={}
    def ev(c,cond,typ,screen,**extra):
        nonlocal tick; tick+=1
        events.append({'session_id':'DRY-'+pid,'participant_id':pid,'reviewer_tier':'trained-reviewer','study_path':'B','sequence':seq,'scenario_id':c['scenario_id'],'condition':cond,'event_type':typ,'event_utc':iso(base,tick),'screen':screen,**extra})
    for c in order:
        sid=c['scenario_id']; cond=condition_for(sid,seq)
        ev(c,cond,'case_opened','Case brief'); ev(c,cond,'screen_opened','Evidence review'); ev(c,cond,'evidence_opened','Evidence review')
        if cond=='ai-first': ev(c,cond,'ai_exposed','Evidence review',ai_visible_before_initial_judgment=True)
        ev(c,cond,'screen_opened','Initial judgment'); ev(c,cond,'initial_judgment_submitted','Initial judgment',initial_judgment='Evidence sufficient',initial_confidence=3,ai_visible_before_initial_judgment=(cond=='ai-first'))
        ev(c,cond,'screen_opened','AI review')
        if cond=='evidence-first': ev(c,cond,'ai_exposed','AI review',ai_visible_before_initial_judgment=False)
        ev(c,cond,'screen_opened','Evidence Passport'); ev(c,cond,'passport_opened','Evidence Passport'); ev(c,cond,'provenance_opened','Evidence Passport',provenance_opened=True)
        ev(c,cond,'screen_opened','Final action'); ev(c,cond,'final_action_submitted','Final action',final_action='Accept AI',final_confidence=3,reason_code='evidence-supports-ai',rationale='SYNTHETIC DRY RUN',provenance_opened=True)
        responses[sid]={'scenario_id':sid,'case_id':c['case_id'],'condition':cond,'initial_judgment':'Evidence sufficient','initial_confidence':3,'provenance_opened':True,'final_action':'Accept AI','final_confidence':3,'reason_code':'evidence-supports-ai','rationale':'SYNTHETIC DRY RUN'}
    return {'synthetic_dry_run':True,'do_not_analyze_as_human_data':True,'participant_id':pid,'reviewer_tier':'trained-reviewer','study_path':'B','sequence':seq,'responses':responses,'events':events}

sessions={'P001':simulate('P001'),'P002':simulate('P002')}
print('✅ P001 Sequence A simulated; P002 Sequence B simulated')


✅ P001 Sequence A simulated; P002 Sequence B simulated


In [5]:
# Cell 5 — Validate event chronology and export schema
import pandas as pd
COLS=['participant_id','reviewer_tier','study_path','sequence','scenario_id','case_id','condition','initial_judgment','initial_confidence','provenance_opened','final_action','final_confidence','reason_code','rationale']
chron=[]
for pid,s in sessions.items():
    for sid in s['responses']:
        es=[e for e in s['events'] if e['scenario_id']==sid]; cond=es[0]['condition']
        ai=[e for e in es if e['event_type']=='ai_exposed']; ini=[e for e in es if e['event_type']=='initial_judgment_submitted']; fin=[e for e in es if e['event_type']=='final_action_submitted']
        assert len(ai)==len(ini)==len(fin)==1
        if cond=='ai-first':
            assert ai[0]['event_utc']<ini[0]['event_utc'] and ai[0]['screen']=='Evidence review' and ai[0]['ai_visible_before_initial_judgment'] is True
        else:
            assert ini[0]['event_utc']<ai[0]['event_utc'] and ai[0]['screen']=='AI review' and ai[0]['ai_visible_before_initial_judgment'] is False
        assert ini[0]['event_utc']<fin[0]['event_utc']; chron.append({'participant_id':pid,'scenario_id':sid,'condition':cond,'chronology_passed':True})
    rows=[{'participant_id':pid,'reviewer_tier':s['reviewer_tier'],'study_path':s['study_path'],'sequence':s['sequence'],**r} for r in s['responses'].values()]
    df=pd.DataFrame(rows)[COLS]
    assert len(df)==12 and df['scenario_id'].nunique()==12 and df['case_id'].nunique()==12
    assert df['condition'].value_counts().to_dict()=={'evidence-first':6,'ai-first':6}
print('✅ 24/24 chronology checks passed')
print('✅ Export schema + completeness passed for P001 and P002')


✅ 24/24 chronology checks passed
✅ Export schema + completeness passed for P001 and P002


In [6]:
# Cell 6 — Save synthetic dry-run artifacts and audit
for pid,s in sessions.items():
    writej(DRY_ROOT/f'SYNTHETIC_neurofhir_review_{pid}.json',s)
    rows=[{'participant_id':pid,'reviewer_tier':s['reviewer_tier'],'study_path':s['study_path'],'sequence':s['sequence'],**r} for r in s['responses'].values()]
    with (DRY_ROOT/f'SYNTHETIC_neurofhir_review_{pid}.csv').open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=COLS); w.writeheader(); w.writerows({c:r.get(c) for c in COLS} for r in rows)
with (DRY_ROOT/'synthetic_chronology_validation.csv').open('w',newline='',encoding='utf-8') as f:
    w=csv.DictWriter(f,fieldnames=list(chron[0].keys())); w.writeheader(); w.writerows(chron)
writej(EVAL_ROOT/'notebook_14c_end_to_end_dry_run_audit.json',{'status':'completed','audited_utc':now(),'notebook':'14C_NeuroFHIR_Review_WISH_End_to_End_Dry_Run_and_Instrumentation_Validation.ipynb','instrumentation':{'ai_first':'before initial judgment','evidence_first':'after initial judgment','one_ai_exposure_per_case':True},'dry_run':{'participants':['P001','P002'],'cases_per_session':12,'case_reviews_validated':24,'chronology_failures':0,'synthetic_exports_are_not_human_data':True},'remaining_manual_gate':'Complete rendered-app click-through as P001 and P002 and confirm four browser downloads.','final_engineering_gate':True})
(DRY_ROOT/'README_DO_NOT_ANALYZE_SYNTHETIC_DRY_RUN.md').write_text('# Synthetic Dry Run\n\nEngineering validation only. Do not include SYNTHETIC files in participant analyses or WISH results.\n',encoding='utf-8')
print('✅ NOTEBOOK 14C AUTOMATED ENGINEERING GATE: TRUE')
print('REMAINING: manual P001/P002 click-through')


✅ NOTEBOOK 14C AUTOMATED ENGINEERING GATE: TRUE
REMAINING: manual P001/P002 click-through


In [8]:
# Cell 7 — Launch final participant app inside Colab for manual P001/P002 dry run
from google.colab import output
PORT=8000
subprocess.run(['bash','-lc',f"pkill -f 'http.server {PORT}' || true"],capture_output=True,text=True)
server=subprocess.Popen(['python','-m','http.server',str(PORT),'--bind','0.0.0.0','--directory',str(PARTICIPANT_ROOT)],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
time.sleep(1.5)
if server.poll() is not None: raise RuntimeError('Participant-app server failed to start.')
print('1) Complete 12 cases as P001 and export CSV + JSON.')
print('2) Reload app.')
print('3) Complete 12 cases as P002 and export CSV + JSON.')
print('Expected hidden allocation: P001=A, P002=B')
print('Do not use these dry-run responses as study data.')
output.serve_kernel_port_as_iframe(PORT,height=900)


1) Complete 12 cases as P001 and export CSV + JSON.
2) Reload app.
3) Complete 12 cases as P002 and export CSV + JSON.
Expected hidden allocation: P001=A, P002=B
Do not use these dry-run responses as study data.


<IPython.core.display.Javascript object>